# <b>1. Data Preparation

### Importing libraries

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

/var/folders/_7/_ccmb0hj6nd6bkr2dvys3bg00000gn/T/ipykernel_2677/3319681691.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## 1. Loading the Document

In [2]:
file_path = '../data/Knowledge.txt'

In [3]:
loader = TextLoader(file_path)
data = loader.load()
print(data[0].page_content)

# TANMAY KHOMANE — PROFESSIONAL KNOWLEDGE BASE

## 1. PROFESSIONAL PROFILE

Tanmay Khomane is a Computer Engineering student at Savitribai Phule Pune University (SPPU), pursuing a Bachelor of Engineering (B.E.) with expected graduation in 2028.

Tanmay works on Artificial Intelligence, Machine Learning, Generative AI, Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), Natural Language Processing (NLP), and Unity game development.

His portfolio includes three AI projects: KAARSHIKA, MIRA, and Tia. His game development projects include The Black Cave: Lost Treasure (Demo) and Dodo's Odyssey.

Tanmay has practical experience building AI pipelines, implementing document retrieval, integrating language models, fine-tuning transformer models, developing Unity games, and publishing Android games.

He prefers hands-on implementation and understanding how software components work internally. His approach emphasizes experimentation, debugging, testing, and improving systems inc

## 2. Chunking the Document

In [4]:
headers_to_split_on = [
    ("#", "section"),
    ("##", "subsection"),
    ("###", "topic"),
]

In [5]:
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=True
)

In [6]:
chunks = splitter.split_text(data[0].page_content)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 79


In [7]:
for i, chunk in enumerate(chunks):
    print(f"\n{'=' * 20} CHUNK {i} {'=' * 20}")
    print(chunk.page_content)


==================== CHUNK 0 ====================
Tanmay Khomane is a Computer Engineering student at Savitribai Phule Pune University (SPPU), pursuing a Bachelor of Engineering (B.E.) with expected graduation in 2028.  
Tanmay works on Artificial Intelligence, Machine Learning, Generative AI, Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), Natural Language Processing (NLP), and Unity game development.  
His portfolio includes three AI projects: KAARSHIKA, MIRA, and Tia. His game development projects include The Black Cave: Lost Treasure (Demo) and Dodo's Odyssey.  
Tanmay has practical experience building AI pipelines, implementing document retrieval, integrating language models, fine-tuning transformer models, developing Unity games, and publishing Android games.  
He prefers hands-on implementation and understanding how software components work internally. His approach emphasizes experimentation, debugging, testing, and improving systems incrementally.

========

## 3. Embedding the Document

In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

In [9]:
document_vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks])

print(f"Number of vectors: {len(document_vectors)}")
print(f"Vector Size: {len(document_vectors[0])}")

Number of vectors: 79
Vector Size: 384


## 4. Store the Vectors in ChromaDB

In [10]:
vector_store = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    collection_name = "Tia_KB_new",
    persist_directory="../vector_store/chroma_db"
)

In [12]:
query = "How does MIRA retrieve mining regulations and use them to generate explanations?"

results = vector_store.max_marginal_relevance_search(
    query,
    k=4,
    fetch_k=12
)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content)


--- Result 1 ---
Metadata: {'topic': 'Project Overview', 'section': 'CATEGORY 1: AI PROJECTS', 'subsection': '4. MIRA — MINE INTELLIGENCE AND RISK ASSESSMENT'}
MIRA, or Mine Intelligence and Risk Assessment, is an AI-powered mine inspection and governance prototype developed in the context of SIH 2026.  
The system is designed to assist with inspection finding classification, risk assessment, and retrieval of relevant mining regulations.  
Its regulatory knowledge source is the Coal Mines Regulations (CMR), 2017.

--- Result 2 ---
Metadata: {'subsection': '4. MIRA — MINE INTELLIGENCE AND RISK ASSESSMENT', 'section': 'CATEGORY 1: AI PROJECTS', 'topic': 'MIRA Regulatory RAG'}
MIRA uses Retrieval-Augmented Generation to retrieve relevant provisions from the Coal Mines Regulations (CMR), 2017.  
Its custom retrieval pipeline uses Dense Passage Retrieval (DPR) embeddings and FAISS vector search.  
The regulatory knowledge base contains approximately 11,726 chunks.  
Retrieved regulatory pa